In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

In [2]:
ff = f"{Path.home()}/gullsInputFiles/runFields/pad_overguide.coords"
outdir = "/fs/project/gaudi.1/crisp/gulls/planets/penny2019_update"

mearth_to_solar = 3.00348959632e-6

In [3]:
rundes = 'penny2019_update'
npf = 10000 # number of planets per file
nff = 20 # number of files per field, i.e., number of subruns

In [12]:
log_mmin = -2.0
log_mmax = 4.0
log_mstep = 0.25

log_amin = -2.0
log_amax = 2.0
log_astep = 0.125

In [13]:
log_m = np.round(np.arange(log_mmin, log_mmax+log_mstep, log_mstep), 2)
log_a = np.round(np.arange(log_amin, log_amax+log_astep, log_astep), 2)
print(len(log_m))
print(len(log_a))

25
33


In [14]:
target_rep = int(np.floor(npf/(len(log_m)*len(log_a))))
print(target_rep)

12


In [15]:
num_lines = int(target_rep * len(log_m) * len(log_a)) # number of lines we actually get in planet file
print(num_lines)

9900


In [16]:
rng = np.random.default_rng()
rnd = rng.random(npf)

In [17]:
fields = pd.read_csv(ff, sep=',', header=0)
# print(fields)
fn = fields['ID_src'].to_numpy()
print(fn)

[129  89  49   9 600 640 680 720 760 800 840 880 920 128  88  48   8 599
 639 679 719 759 799 839 879 919 127  87  47   7 598 638 678 718 758 798
 838 878 918 126  86  46   6 597 637 677 717 757 797 837 877 917 125  85
  45   5 596 636 676 716 756 796 836 876 916 124  84  44   4 595 635 675
 715 755 795 835 875 915 123  83  43   3 594 634 674 714 754 794 834 874
 914 122  82  42   2 593 633 673 713 753 793 833 873 913  41   1 592 632
 672  40   0 591 631 671  60  20 611 651 691  61  21 612 652 692  62  22
 613 653 693]


In [18]:
for subrun in range(nff):
    for field in fn:
        name = f"{rundes}.planets.{field}.{subrun}"
        
        # orbital phase (deg)
        p_array = 360.0 * rng.random(num_lines)
        
        # Isotropic inclination (signed degrees)
        rnd = rng.random(num_lines)
        arccos_arg = np.where(rnd < 0.5, 2 * rnd, 2 - 2 * rnd)
        safe_arg = np.clip(arccos_arg, -1.0, 1.0)
        angle = np.arccos(safe_arg)
        signed_angle = np.where(rnd < 0.5, angle, -angle)
        inc_array = 180 * signed_angle / np.pi
        
        mass_array = []
        sma_array = []
        for rep in range(target_rep):
            for m in log_m:
                for a in log_a:
                    mass_array.append(mearth_to_solar*10**m)
                    sma_array.append(10**a)
        
        sr_df = pd.DataFrame(list(zip(mass_array, sma_array, inc_array, p_array)))
        sr_df.to_csv(f"{outdir}/{name}", sep=' ', header=False, index=False)

In [24]:
# # Orbital phase (deg)
# p_array = 360.0 * rng.random(npf)

In [25]:
# # Isotropic inclination (signed degrees)
# arccos_arg = np.where(rnd < 0.5, 2 * rnd, 2 - 2 * rnd)
# safe_arg = np.clip(arccos_arg, -1.0, 1.0)
# angle = np.arccos(safe_arg)
# signed_angle = np.where(rnd < 0.5, angle, -angle)
# inc_array = 180 * signed_angle / np.pi